In [ ]:
import torch

from dataset_loaders.latent_normalizer import LatentNormalizer
from utils.checkpoints import load_ae_from_path, load_cspn_from_path, load_from_wandb
from utils.visualisation import show, plot_latent_comparison, plot_latent_comparison_multiclass
from utils.config import DatasetConfig
from dataset_loaders import build_data_loaders
from pathlib import Path


In [ ]:
ae_path = Path("../checkpoints/autoencoder/cub200.pt")
ae = load_ae_from_path(ae_path)

normalizer = LatentNormalizer()


In [ ]:
dataset_cfg = DatasetConfig(
    name="mnist",
    channels=3,
    height=128,
    width=128,
    num_classes=200,
    num_workers=1
)

dataloader, _ = build_data_loaders(dataset_cfg, batch_size=64)
normalizer.fit(ae, dataloader, torch.device("cpu"))


In [ ]:
cspn_psi_path = load_from_wandb("cspn_cub200_psinet")
cspn_psi = load_cspn_from_path(cspn_psi_path, device=torch.device("cpu"))

In [ ]:
label = 0
sample_labels = torch.tensor([label] * 3)

with torch.no_grad():
    samples_psi = cspn_psi.sample(sample_labels)
    denormalized_psi = normalizer.denormalize(samples_psi)
    sampled_images_psi = ae.decode(samples_psi)

show(sampled_images_psi, f"Samples from PSINet CSPN with label {label}")

In [ ]:
all_labels = torch.arange(10).repeat_interleave(5)

with torch.no_grad():
    samples = cspn_custom.sample(all_labels)
    sampled_images = ae.decode(samples)
    samples_psi = cspn_psi.sample(all_labels)
    sampled_images_psi = ae.decode(samples_psi)

show(sampled_images, "Samples from CSPN for all labels", width=5)
show(sampled_images_psi, "Samples from PSINet CSPN for all labels", width=5)

In [ ]:
sample_multi_labels = torch.arange(10).repeat_interleave(100)
with torch.no_grad():
    samples_custom = cspn_custom.sample(sample_multi_labels)
    #samples_psi = cspn_psi.sample(sample_multi_labels)
    samples_psi_sep = []
    for cls in range(10):
        cls_labels = torch.tensor([cls] * 100)
        samples_psi_cls = cspn_psi.sample(cls_labels)
        samples_psi_sep.append(samples_psi_cls)
    samples_psi = torch.cat(samples_psi_sep, dim=0)
plot_latent_comparison_multiclass(samples_custom, samples_psi, sample_multi_labels,
                                  name_a="Custom CSPN", name_b="PSI-Net CSPN")
